# Question Answering

In [1]:
documents = [
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The Earth revolves around the Sun once every 365 days.",
    "The chemical symbol for gold is Au.",
    "Light travels at a speed of approximately 299,792 kilometers per second.",
    "The human body has 206 bones.",
    "Photosynthesis occurs in the chloroplasts of plant cells."
]

In [2]:
import re
import numpy as np
from collections import Counter

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

tokenized_docs = [preprocess(doc) for doc in documents]
vocab = sorted(list(set([w for doc in tokenized_docs for w in doc])))

def vectorize(doc_tokens, vocab):
    vec = [0]*len(vocab)
    count = Counter(doc_tokens)
    for i, w in enumerate(vocab):
        vec[i] = count.get(w, 0)
    return vec

doc_vectors = [vectorize(doc, vocab) for doc in tokenized_docs]


### Similarity Based Retrieval

In [3]:
def cosine_sim(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    if np.linalg.norm(vec1)==0 or np.linalg.norm(vec2)==0:
        return 0
    return np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))

def retrieve(query, doc_vectors, documents, vocab):
    query_tokens = preprocess(query)
    query_vec = vectorize(query_tokens, vocab)
    sims = [cosine_sim(query_vec, doc_vec) for doc_vec in doc_vectors]
    best_idx = np.argmax(sims)
    return documents[best_idx], sims[best_idx]


### Conditional Answering

In [4]:
def answer_question(query, retrieved_doc):
    query = query.lower()
    if "who" in query:
        if "sun" in retrieved_doc.lower():
            return "The Sun"
        elif "human body" in retrieved_doc.lower():
            return "The human body"
    elif "what" in query:
        if "boils" in retrieved_doc.lower():
            return "Water boils at 100 degrees Celsius"
        elif "symbol" in retrieved_doc.lower():
            return "Au"
        elif "photosynthesis" in retrieved_doc.lower():
            return "Photosynthesis occurs in chloroplasts"
        elif "speed" in retrieved_doc.lower():
            return "299,792 km/s"
        elif "bones" in retrieved_doc.lower():
            return "206 bones"
    elif "where" in query:
        if "chloroplast" in retrieved_doc.lower():
            return "In plant cells"
    return retrieved_doc  # fallback: return the full document


### Simple QA / RAG

In [5]:
def simple_rag(query):
    retrieved_doc, sim = retrieve(query, doc_vectors, documents, vocab)
    ans = answer_question(query, retrieved_doc)
    print(f"Query: {query}")
    print(f"Retrieved Document (Sim={sim:.2f}): {retrieved_doc}")
    print(f"Answer: {ans}\n")

# Example queries
simple_rag("At what temperature does water boil?")
simple_rag("How fast does light travel?")
simple_rag("How many bones are in the human body?")
simple_rag("Where does photosynthesis occur?")
simple_rag("What is the chemical symbol for gold?")


Query: At what temperature does water boil?
Retrieved Document (Sim=0.61): Water boils at 100 degrees Celsius at standard atmospheric pressure.
Answer: Water boils at 100 degrees Celsius

Query: How fast does light travel?
Retrieved Document (Sim=0.30): Light travels at a speed of approximately 299,792 kilometers per second.
Answer: Light travels at a speed of approximately 299,792 kilometers per second.

Query: How many bones are in the human body?
Retrieved Document (Sim=0.73): The human body has 206 bones.
Answer: The human body has 206 bones.

Query: Where does photosynthesis occur?
Retrieved Document (Sim=0.35): Photosynthesis occurs in the chloroplasts of plant cells.
Answer: In plant cells

Query: What is the chemical symbol for gold?
Retrieved Document (Sim=0.93): The chemical symbol for gold is Au.
Answer: Au

